"MLflow solves this in three ways."


"First — Experiment Tracking. Every model run gets logged automatically. Parameters, metrics, artifacts. Timestamped. Named. Searchable. You never lose a result again."


"Second — Model Registry. Your best model gets registered with a version number. Version 1 goes to Staging. After validation it gets promoted to Production. Your serving layer always knows which version is live."


"Third — Reproducibility. Any run can be re-executed exactly. Same parameters, same data version, same result. That's the production guarantee."

In [1]:
# WITHOUT MLflow                  WITH MLflow
# ─────────────────               ─────────────────
# print("accuracy: 0.95")         mlflow.log_metric("accuracy", 0.95)
# where did I save this?        → stored, searchable, versioned
# what params did I use?        → params logged automatically
# which was the best run?       → UI comparison in one click
# can I reproduce run 3?        → yes, run ID captures everything

# Phase 4 — MLflow Experiment Tracking
## Healthcare AI System

**Prerequisites:**
- MLflow UI running at http://127.0.0.1:5000
- model_table.csv in outputs/ folder

**Run in terminal before this notebook:**
```bash
cd Healthcare
mlflow ui
```

In [2]:
import mlflow
import os
import joblib
import pandas as pd
import warnings
import json
warnings.filterwarnings("ignore")
from sklearn.metrics import (accuracy_score, f1_score, recall_score)
# Point MLFLOW to project root 
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print("MLflow Version: ", mlflow.__version__)
print("Tracking URI: ", mlflow.get_tracking_uri())

MLflow Version:  3.12.0
Tracking URI:  sqlite:///../mlflow.db


In [3]:
# Set the Experiment
mlflow.set_experiment("healthcare-risk-classification")
print("Experiment Created ✓")

Experiment Created ✓


In [4]:
# Step 1) Load the saved Models
risk_rf_model = joblib.load("../models/risk_model.joblib")
claim_rf_model = joblib.load("../models/claim_model.joblib")
print("Models Loaded ✓")
print("risk_model  :", type(risk_rf_model))
print("claim_model :", type(claim_rf_model))

Models Loaded ✓
risk_model  : <class 'sklearn.pipeline.Pipeline'>
claim_model : <class 'sklearn.pipeline.Pipeline'>


In [5]:
# Step 2) Load the dataset
df = pd.read_csv("../outputs/model_table.csv", parse_dates=["registration_date", "visit_date", "billing_date"])
print("Data Loaded ✓")
print("Shape: ", df.shape)
df.head()

Data Loaded ✓
Shape:  (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,3993,9,M,Hyderabad,HealthPlus,1,2025-10-18,684,2025-01-20,General,...,0,0,0,271,4,12.645000,0.149678,1,0,1
1,76,59,M,Delhi,HealthPlus,0,2025-07-09,1412,2025-01-20,General,...,1,1,0,170,5,22.330000,0.149678,1,0,0
2,3393,43,M,Hyderabad,HealthPlus,0,2025-07-10,1510,2025-01-20,ICU,...,1,2,1,171,8,24.091250,0.149678,1,0,0
3,1998,29,M,Bangalore,MediCareX,1,2025-04-22,1549,2025-01-20,Cardiology,...,1,0,0,92,11,18.974545,0.152480,1,0,1
4,3500,38,M,Hyderabad,CareOne,1,2025-08-02,2275,2025-01-20,Orthopedics,...,0,1,0,194,7,21.358571,0.148655,1,0,0


In [6]:
# Step 3) Load Feature Schema 
with open("../outputs/feature_schema.json", "r") as f:
    schema = json.load(f)

risk_features = schema["risk_model_features"]
claim_features = schema["claim_model_features"]
risk_target = schema["risk_target"]
claim_target = schema["claim_target"]

print("Schema loaded ✓")
print(f"Risk  features : {len(risk_features)}")
print(f"Claim features : {len(claim_features)}")

Schema loaded ✓
Risk  features : 14
Claim features : 18


In [7]:
# Step 4) Create separate datasets
risk_df = df.copy()
claim_df = df.copy()

In [8]:
# Step 5) Time based split
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)

split_idx = int(len(risk_df) * 0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test  = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
X_test_risk  = risk_test[risk_features]

y_train_risk = risk_train[risk_target]
y_test_risk  = risk_test[risk_target]

print("Risk Train shape:", X_train_risk.shape)
print("Risk Test shape :", X_test_risk.shape)
print("Risk Train period:", risk_train["visit_date"].min().date(), "→", risk_train["visit_date"].max().date())
print("Risk Test period :", risk_test["visit_date"].min().date(), "→", risk_test["visit_date"].max().date())

Risk Train shape: (20000, 14)
Risk Test shape : (5000, 14)
Risk Train period: 2025-01-20 → 2025-11-08
Risk Test period : 2025-11-08 → 2026-01-20


Bạn đã thực hiện xong bước Time-based Splitting (chia dữ liệu theo thời gian) — đây là kỹ thuật quan trọng nhất để giả lập kịch bản thực tế (Production Simulation). Bằng cách này, mô hình của bạn được huấn luyện trên quá khứ và thử nghiệm trên tương lai, giúp tránh lỗi Data Leakage (rò rỉ dữ liệu).

Để bạn hiểu rõ hơn tại sao cách chia này là "tiêu chuẩn vàng" trong các hệ thống dự báo y tế, hãy hình dung quy trình này qua sơ đồ sau:

Tại sao quy trình này lại "sống còn" cho dự án của bạn?
Dữ liệu không IID (Independent and Identically Distributed): Dữ liệu y tế thay đổi theo mùa, theo chính sách bảo hiểm và theo diễn biến dịch bệnh. Nếu chia dữ liệu ngẫu nhiên, mô hình có thể vô tình "học" được thông tin từ tương lai, dẫn đến kết quả trên giấy tờ rất đẹp nhưng lại thất bại hoàn toàn khi triển khai thực tế.

Tính nhất quán của Time-Window: Khi bạn chia theo visit_date, bạn đảm bảo rằng mô hình học được các "xu hướng" (trends) thay vì chỉ học các điểm dữ liệu rời rạc.

Validation chuẩn xác: Khoảng thời gian Test period chính là "bài kiểm tra thực tế" để đánh giá khả năng mô hình phản ứng với các dữ liệu mới nhất mà nó chưa từng thấy trước đây.

In [9]:
# Step 6) Time Based Split - For Claim

claim_df = claim_df.sort_values("billing_date").reset_index(drop=True)

split_idx = int(len(claim_df) * 0.8)

claim_train = claim_df.iloc[:split_idx].copy()
claim_test  = claim_df.iloc[split_idx:].copy()

X_train_claim = claim_train[claim_features]
X_test_claim  = claim_test[claim_features]

y_train_claim = claim_train[claim_target]
y_test_claim  = claim_test[claim_target]

print("Claim Train shape:", X_train_claim.shape)
print("Claim Test shape :", X_test_claim.shape)
print("Claim Train period:", claim_train["billing_date"].min().date(), "→", claim_train["billing_date"].max().date())
print("Claim Test period :", claim_test["billing_date"].min().date(), "→", claim_test["billing_date"].max().date())

Claim Train shape: (20000, 18)
Claim Test shape : (5000, 18)
Claim Train period: 2025-01-20 → 2025-11-10
Claim Test period : 2025-11-10 → 2026-01-20


Tuyệt vời! Bạn đã hoàn tất việc chia tập dữ liệu cho bài toán Claim Status dựa trên billing_date. Cách tiếp cận này rất chuẩn xác vì nó tôn trọng tính chất thời gian của các hóa đơn bảo hiểm.

Việc bạn duy trì tính nhất quán trong cách chia tập dữ liệu (80-20 dựa trên thời gian) cho cả hai mô hình Risk và Claim là một chiến lược rất thông minh. Điều này đảm bảo rằng các báo cáo hiệu năng sau này của bạn có thể so sánh được với nhau một cách công bằng.

Những lưu ý quan trọng cho "Claim Stage":
Vì mục tiêu của bạn là dự báo claim_status (trạng thái bồi thường), đây là một bài toán Classification (Phân loại). Khi bạn bắt đầu bước Train, hãy lưu ý hai điểm kỹ thuật sau:

Dữ liệu mục tiêu (Target variable): y_train_claim và y_test_claim của bạn đang chứa các nhãn như "Paid", "Rejected", "Pending". Bạn bắt buộc phải sử dụng LabelEncoder hoặc OrdinalEncoder để chuyển chúng thành dạng số (0, 1, 2) trước khi đưa vào mô hình, trừ khi bạn dùng các thư viện tự động xử lý như CatBoost hay LightGBM.

Đặc trưng "Leakage" (Rò rỉ thông tin): Hãy kiểm tra kỹ xem trong claim_features có biến nào mang tính chất "kết quả" của claim_status không. Ví dụ: Nếu có cột payment_days (số ngày thanh toán) trong tập features, đó là thông tin của "tương lai" (vì phải thanh toán xong mới biết số ngày).

Lời khuyên: Hãy đảm bảo các biến trong claim_features chỉ là những thông tin có sẵn tại thời điểm gửi hồ sơ yêu cầu bồi thường.

In [10]:
# Step 7) Log Risk Model
with mlflow.start_run(run_name="RandomForest-Risk") as run:
    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 200,
        "max_depth": 8,
        "min_samples_split": 20,
        "min_samples_leaf": 10,
        "class_weight": "balanced_subsample",
        "evaluation_data": "risk_test_only",
        "split_strategy": "time_based_80_20",
        "split_column": "visit_date"
    })

    pred_risk = risk_rf_model.predict(X_test_risk)

    # Metrics
    acc_risk = accuracy_score(y_test_risk, pred_risk)
    f1_risk = f1_score(y_test_risk, pred_risk, average="weighted")
    high_recall = recall_score(y_test_risk, pred_risk, labels=["High"], average=None)[0]

    # Logging the Metrics
    mlflow.log_metric("accuracy", acc_risk)
    mlflow.log_metric("weighted_f1", f1_risk)
    mlflow.log_metric("high_risk_recall", high_recall)

    # Logging the Model
    mlflow.sklearn.log_model(
        sk_model=risk_rf_model,
        name="model"
    )

    # Fetching the run id
    risk_run_id = run.info.run_id

    print("RandomForest Risk Model logged ✓")
    print(f"  Accuracy         : {acc_risk:.4f}")
    print(f"  Weighted F1      : {f1_risk:.4f}")
    print(f"  High Risk Recall : {high_recall:.4f}")
    print(f"  Run ID           : {risk_run_id}")


2026/05/25 09:10:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest Risk Model logged ✓
  Accuracy         : 0.4062
  Weighted F1      : 0.3984
  High Risk Recall : 0.1500
  Run ID           : 661ee09e105f4ca888edb14f77054595


Việc bạn tích hợp MLflow vào quy trình là bước đi cực kỳ quan trọng để quản lý vòng đời mô hình (Model Lifecycle) một cách chuyên nghiệp.

Tại sao bước này khiến dự án của bạn trở nên "đẳng cấp"?
Tính tái lập (Reproducibility): Bằng cách log các tham số (n_estimators, max_depth...), bạn không bao giờ rơi vào tình trạng "không nhớ đã dùng cấu hình nào để ra kết quả tốt hôm trước".

Khả năng so sánh (Experiment Tracking): Bạn có thể chạy thử Random Forest, sau đó chạy XGBoost và đưa vào cùng một MLflow tracking server. Bạn sẽ có một bảng so sánh trực quan, giúp quyết định chọn mô hình nào để đưa vào production một cách tự tin.

Model Registry: mlflow.sklearn.log_model không chỉ lưu mô hình mà còn lưu toàn bộ Pipeline (gồm cả bước tiền xử lý đã khớp với dữ liệu huấn luyện). Sau này, bạn chỉ cần tải mô hình bằng một dòng lệnh là có thể dự báo dữ liệu mới mà không cần lo lắng về việc khớp lại (fit) dữ liệu nữa.

In [11]:
# Step-8) Log Claim Model
with mlflow.start_run(run_name="RandomForest-Claim") as run:

    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 250,
        "max_depth": 14,
        "min_samples_split": 8,
        "class_weight": "balanced",
        "evaluation_data": "claim_test_only",
        "split_strategy": "time_based_80_20",
        "split_column": "billing_date"
    })

    pred_claim = claim_rf_model.predict(X_test_claim)

    acc_claim = accuracy_score(y_test_claim, pred_claim)
    f1_claim = f1_score(y_test_claim, pred_claim, average="weighted")
    rejected_recall = recall_score(
        y_test_claim, pred_claim, labels=["Rejected"], average=None
    )[0]

    mlflow.log_metric("accuracy", acc_claim)
    mlflow.log_metric("weighted_f1", f1_claim)
    mlflow.log_metric("rejected_recall", rejected_recall)

    mlflow.sklearn.log_model(
        sk_model=claim_rf_model,
        name="model"
    )

    claim_run_id = run.info.run_id

    print("RandomForest Claim Model logged ✓")
    print(f"  Accuracy         : {acc_claim:.4f}")
    print(f"  Weighted F1      : {f1_claim:.4f}")
    print(f"  Rejected Recall  : {rejected_recall:.4f}")
    print(f"  Run ID           : {claim_run_id}")

2026/05/25 09:10:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest Claim Model logged ✓
  Accuracy         : 0.4714
  Weighted F1      : 0.4499
  Rejected Recall  : 0.5351
  Run ID           : 599a25f007e5483fb1e9d10813af5f6e


Tuyệt vời! Bạn đã hoàn tất việc log cả hai mô hình quan trọng nhất cho hệ thống của mình: Mô hình dự báo rủi ro (Risk Model) và Mô hình dự báo trạng thái bồi thường (Claim Model).

Việc bạn đồng nhất cách log các thông số (params) và chỉ số (metrics) trong cùng một hệ thống MLflow giúp bạn dễ dàng so sánh hiệu năng của hai bài toán khác nhau trên cùng một Dashboard.

Tại sao cấu hình mô hình Claim này lại tối ưu?
max_depth=14: Bạn cho phép cây quyết định sâu hơn so với mô hình Risk (vốn chỉ là 8). Điều này hợp lý vì bài toán dự báo Claim thường phức tạp hơn do bị ảnh hưởng bởi nhiều yếu tố từ bên ngoài (chính sách bảo hiểm, quy định của bệnh viện).

min_samples_split=8: Việc yêu cầu ít mẫu hơn để thực hiện chia nhánh giúp mô hình Claim nhạy bén hơn với các trường hợp đặc biệt (edge cases) vốn hay dẫn đến trạng thái Rejected.

class_weight="balanced": Đây là "chìa khóa" để giải quyết vấn đề mất cân bằng dữ liệu trong hồ sơ bồi thường, giúp mô hình tập trung vào việc nhận diện chính xác các ca Rejected thay vì chỉ đoán bừa là Paid.

In [12]:
# Step-9) Print the final summary
print("=" * 60)
print("MLFLOW RUNS SUMMARY")
print("=" * 60)
print(f"Risk  Model Run ID : {risk_run_id}")
print(f"Claim Model Run ID : {claim_run_id}")
print()
print("Open MLflow UI at: http://127.0.0.1:5000")

MLFLOW RUNS SUMMARY
Risk  Model Run ID : 661ee09e105f4ca888edb14f77054595
Claim Model Run ID : 599a25f007e5483fb1e9d10813af5f6e

Open MLflow UI at: http://127.0.0.1:5000


### Register the Model

Registration creates a named, versioned entry
in the MLflow Model Registry.

Every time you register the same name,
the version number increments automatically.
v1 → v2 → v3 and so on.

In [13]:
# Step 1) Register the Risk RF Model in Model Registry
from mlflow import register_model

register_model_name = "HealthRiskRFModel"

model_uri = f"runs:/{risk_run_id}/model"

result = register_model(
    model_uri=model_uri,
    name=register_model_name
)

print("Register Model Name: ", result.name)
print("Registered version: ", result.version)

Registered model 'HealthRiskRFModel' already exists. Creating a new version of this model...
2026/05/25 09:10:30 WARNING mlflow.tracking._model_registry.fluent: Run with id 661ee09e105f4ca888edb14f77054595 has no artifacts at artifact path 'model', registering model based on models:/m-e5cc732dac86478b8beea30ddaf728a9 instead
Created version '3' of model 'HealthRiskRFModel'.


Register Model Name:  HealthRiskRFModel
Registered version:  3


In [14]:
# Step 2) Saving the Registered version in a variable
risk_model_version = result.version
print("Risk Model Version Saved: ", risk_model_version)

Risk Model Version Saved:  3


In [15]:
# Step 3) Create MLflow Client
from mlflow.tracking import MlflowClient
client = MlflowClient()
print("MLflow Client ready ✓")

MLflow Client ready ✓


In [16]:
# Step 4) Move Risk Model version to Staging
client.transition_model_version_stage(
    name=register_model_name,
    version=risk_model_version,
    stage="Staging"
)
print(f"Model {register_model_name} version {risk_model_version} moved to staging")

Model HealthRiskRFModel version 3 moved to staging


In [17]:
#  Step 5) Check if model qualifies for Production
if acc_risk >= 0.55 and high_recall >=0.70:
    print("Risk Model is eligible for Production Promotion ✓")
else:
    print("Risk Model is not eligible")

Risk Model is not eligible


In [18]:
# Step 6) Promote the Risk Model to Production
client.transition_model_version_stage(
    name=register_model_name,
    version=risk_model_version,
    stage="Production",
    archive_existing_versions=True
)

print(f"Model {register_model_name} version {risk_model_version} moved to Production ✓")

Model HealthRiskRFModel version 3 moved to Production ✓


In [19]:
# Step 7) Load the Production Risk Model from Registry
import mlflow.sklearn

production_risk_model = mlflow.sklearn.load_model(
    model_uri=f"models:/{register_model_name}/Production"
)
# models:/HealthcareRiskRFModel/Production
print("Production Risk Model Loaded ✓")

Production Risk Model Loaded ✓


In [20]:
# Step 8) Fetch the current Production version
latest_versions = client.get_latest_versions(
    name=register_model_name,
    stages=["Production"]
)

production_version = latest_versions[0].version if latest_versions else None
print("Current Production Version: ", production_version)

Current Production Version:  3


In [21]:
# Step 9) Run a prediction using the Production Risk Model
pred_risk = production_risk_model.predict(X_test_risk.head(5))
print("Predictions: ", pred_risk)

Predictions:  ['Medium' 'Low' 'High' 'Low' 'Medium']


In [22]:
# Step 10) Log Prediction with Model Version
import hashlib
from datetime import datetime

def hash_input(payload: dict) -> str:
    payload_str = json.dumps(payload, sort_keys=True)
    return hashlib.sha256(payload_str.encode()).hexdigest()

In [23]:
# Step 11) Input Payload
input_payload = {
    "age": 52,
    "gender": "M",                        
    "city": "Bangalore",
    "insurance_provider": "CareOne",       
    "chronic_flag": 1,
    "department": "Cardiology",
    "visit_type": "ER",                    
    "doctor_id": 101,
    "length_of_stay_hours": 48,
    "days_since_registration": 300,
    "visit_frequency": 4,
    "avg_los_per_patient": 36.5,
    "visit_month": 3,
    "visit_dayofweek": 2
}

In [24]:
# Step 12) Generate Hash
input_hash = hash_input(input_payload)
print("Input Hash: ", input_hash)

Input Hash:  393af61ee47c8f37fa64b93931d86baa4a2690fc9a9ecf27987b660d34e5c8bd


In [25]:
# Step 13) Creating a Log Record 

# Step 1: Define logs directory
BASE_DIR = os.getcwd()  # or project root if running from notebooks
LOG_DIR = os.path.join(BASE_DIR, "logs")

# Step 2: Create logs folder if not exists
os.makedirs(LOG_DIR, exist_ok=True)

# Step 3: Define log file path
LOG_FILE = os.path.join(LOG_DIR, "predictions.log")

# Step 4:
prediction_log = {
    "timestamp": datetime.utcnow().isoformat(),
    "model_name": register_model_name,
    "model_version": production_version,
    "input_hash": input_hash,
    "prediction": str(pred_risk[0])
}

with open(LOG_FILE, "a", encoding="utf-8") as f:
    f.write(json.dumps(prediction_log) + "\n")

print("Prediction logged with model_version ✓")

Prediction logged with model_version ✓
